# `ptof_obs_nightly_baseline`

## What this notebook does
Recomputes the statistical baseline that detection depends on to tell "normal" from
"anomalous": per-capability, per-field response presence rate, and an ETL-duration baseline.
Nothing in this notebook detects anything itself — it produces the reference points
`ptof_obs_mal_output` and `ptof_obs_liveness_detection` compare live data against.

## Position in the pipeline
- **Separate scheduled job** (`obs_nightly_baseline`) — runs nightly, independently of
  `obs_fresh_scan`. Not one of `obs_fresh_scan`'s tasks.
- **Upstream:** reads `v_llm_bronze`, `v_etl_bronze` (built by `ptof_obs_bronze_projection`) and
  `capability_registry` (human-curated by `ptof_obs_setup_seed`).
- **Downstream:** `ptof_obs_mal_output` reads `response_field_baseline` to detect schema drift
  (a field that used to reliably appear has gone missing). `ptof_obs_liveness_detection` reads
  `etl_duration_baseline` for `etl_run_slow`.

## Why baselines are computed nightly, separately from detection
The queries do a `CREATE OR REPLACE` over a 30-day rolling window — expensive relative to the
hourly detection queries. Splitting this into its own nightly job means detection runs stay cheap
and compare against a stable reference point.

## Tables/views touched
- **Reads:** `v_llm_bronze`, `v_etl_bronze`, `capability_registry`
- **Writes:** `response_field_baseline` (per-capability, per-field presence rate),
  `etl_duration_baseline`

## ETL-duration baseline (added 2026-09-18)
- `etl_duration_baseline` — median + MAD of `v_etl_bronze.duration_seconds` per table_or_view x
  task_name (30-day window, `status='success'` only, floor n>=50, no fallback tier). Feeds
  `etl_run_slow` in `ptof_obs_liveness_detection` (CRITICAL — promoted 2026-09-21, see
  `threshold_basis`). `upper_bound_s` floored at `median_duration_s * 1.5` (added 2026-09-22)
  so a degenerate/near-zero-MAD stratum can't flag any run a second over its own median —
  0 such strata exist in prod as of that date, so this is precautionary, not an active fix.

## Dropped baselines (prod migration 2026-09-10)
- `capability_latency_baseline` — no `latency_ms` in prod. Phase 2 via dev enrichment.

## Dropped tables (2026-09-21, detector value audit + dead-code audit)
- `write_lag_baseline` — its only consumer, `write_lag_anomalies`, was retired in the same
  audit (write_lag_s had zero variance across 10,721 rows/30 days — no information content).
  This baseline table went stranded the moment that detector was deleted; removed here too.

In [ ]:
%sql
-- response_field_baseline: per-capability, per-field presence rate computed nightly over a
-- 30-day rolling window. Supports ptof_obs_mal_output's response_schema_drift detector.
-- Prod context: all rows in ai_shift_outputs are successful outputs (no success/error columns),
-- so no success or credential-fastfail filter is needed. response_parsed is aliased from
-- content in the view, which is 100% valid JSON in prod.
-- Floor of 20 eligible rows per capability: prevents thin data from creating a noisy baseline.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.response_field_baseline AS
WITH eligible AS (
    -- non-blank outputs from active capabilities in the last 30 days
    SELECT b.capability, b.response_parsed
    FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
    JOIN mq_gmdf_dev.oil_obs.capability_registry r
      ON r.capability = b.capability AND r.active = true
    WHERE b.called_at >= current_timestamp() - INTERVAL 30 DAYS
      AND b.is_blank_output = false
),
row_counts AS (
    -- per-capability row counts, filtered to >= 20 to avoid thin baselines
    SELECT capability, count(*) AS n_rows FROM eligible GROUP BY capability HAVING count(*) >= 20
),
field_counts AS (
    -- explode each output's JSON keys and count how many rows each field appears in
    SELECT e.capability, k.key AS field_name, count(*) AS baseline_present
    FROM eligible e
    LATERAL VIEW explode(from_json(cast(e.response_parsed AS STRING), 'map<string,string>')) k AS key, val
    WHERE e.capability IN (SELECT capability FROM row_counts)
    GROUP BY e.capability, k.key
)
SELECT f.capability, f.field_name, f.baseline_present,
       r.n_rows AS baseline_total,
       f.baseline_present * 1.0 / r.n_rows AS baseline_presence_rate,
       current_timestamp() AS computed_at
FROM field_counts f
JOIN row_counts r ON r.capability = f.capability;

In [ ]:
%sql
-- etl_duration_baseline: robust (median + MAD) baseline of v_etl_bronze.duration_seconds, per
-- table_or_view x task_name, over a 30-day rolling window. Feeds ptof_obs_liveness_detection's
-- etl_run_slow (CRITICAL, promoted 2026-09-21 -- see threshold_basis). Filtered to
-- status='success' only -- a failed run's duration isn't a meaningful "how long does this
-- normally take" data point (it may have errored out early or hung before failing), and
-- failures are already covered by the separate etl_pipeline_health detector.
-- Floor of 50 rows per (table_or_view, task_name) stratum, same rationale as write_lag_baseline.
-- No fallback tier here (unlike write_lag_baseline) -- table_or_view x task_name is already the
-- coarsest grouping that makes sense (durations aren't comparable across different tables/tasks),
-- so a stratum under 50 rows simply has no baseline yet rather than a misleading broader one.
--
-- upper_bound_s floor (added 2026-09-22, senior-architect review): without it, a stratum with
-- mad_duration_s = 0 (or near-0) collapses upper_bound_s toward median_duration_s, so any run a
-- second or two over its own median would get flagged CRITICAL -- the same degenerate-MAD shape
-- that got write_lag_anomalies retired entirely (its write_lag_s was constant 0 across all
-- 10,721 rows, making that MAD-based guard untunable regardless of floor). Diagnostic run
-- 2026-09-22 against prod (see ptof_obs_liveness_detection.ipynb, cell after etl_run_slow) found
-- 0 strata with mad_duration_s < 1 today, so this floor is precautionary rather than closing an
-- active false-positive. greatest(..., median_duration_s * 1.5) chosen as a generous margin
-- consistent with this pipeline's other thresholds (e.g. etl_table_staleness's 4-6x,
-- capability_silence's 3-7x) -- open to revisiting once a real degenerate stratum, if one ever
-- appears, gives something concrete to tune against.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_duration_baseline AS
WITH eligible AS (
    SELECT table_or_view, task_name, duration_seconds
    FROM mq_gmdf_dev.oil_obs.v_etl_bronze
    WHERE run_timestamp >= current_timestamp() - INTERVAL 30 DAYS
      AND status = 'success'
),
med AS (
    SELECT table_or_view, task_name,
           count(*)                                     AS n,
           percentile_approx(duration_seconds, 0.5)      AS med_s
    FROM eligible
    GROUP BY table_or_view, task_name
)
SELECT
    e.table_or_view,
    e.task_name,
    m.n,
    m.med_s                                                        AS median_duration_s,
    percentile_approx(abs(e.duration_seconds - m.med_s), 0.5)      AS mad_duration_s,
    greatest(
        m.med_s + 5 * 1.4826 * percentile_approx(abs(e.duration_seconds - m.med_s), 0.5),
        m.med_s * 1.5
    )                                                               AS upper_bound_s,
    current_timestamp()                                            AS computed_at
FROM eligible e
JOIN med m USING (table_or_view, task_name)
GROUP BY e.table_or_view, e.task_name, m.n, m.med_s
HAVING m.n >= 50;